# Semantic Model Dimensions

This notebook creates the physical `dim_date` and `dim_resource` Delta tables required by the Challenge 5 Direct Lake semantic model. It derives both dimensions from the validated Gold tables and does not create synthetic Fabric capacity telemetry.


In [ ]:
from functools import reduce

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F


## Date Dimension

Build one continuous calendar spanning the cost, operational, agent, and resource snapshot dates currently present in Gold.


In [ ]:
gold_cost = spark.read.format("delta").load("Tables/dbo/gold_cost_summary")
gold_operations = spark.read.format("delta").load("Tables/dbo/gold_operational_metrics")
gold_agents = spark.read.format("delta").load("Tables/dbo/gold_agent_analytics")
gold_resources = spark.read.format("delta").load("Tables/dbo/gold_resource_inventory")

date_frames: list[DataFrame] = [
    gold_cost.select(F.col("period_start").alias("date_value")),
    gold_cost.select(F.col("period_end").alias("date_value")),
    gold_operations.select(F.col("metric_date").alias("date_value")),
    gold_agents.select(F.col("interaction_date").alias("date_value")),
    gold_resources.select(F.col("snapshot_date").alias("date_value")),
]
all_dates = reduce(DataFrame.unionByName, date_frames).where(F.col("date_value").isNotNull())
date_bounds = all_dates.agg(
    F.min("date_value").alias("min_date"),
    F.max("date_value").alias("max_date"),
).first()

if date_bounds.min_date is None or date_bounds.max_date is None:
    raise ValueError("No populated Gold date columns were found. Run notebooks 03 and 04 first.")

dim_date = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(date_bounds.min_date),
                F.lit(date_bounds.max_date),
                F.expr("INTERVAL 1 DAY"),
            )
        ).alias("Date")
    )
    .withColumn("Year", F.year("Date"))
    .withColumn("Quarter", F.quarter("Date"))
    .withColumn("Month", F.month("Date"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("Day", F.dayofmonth("Date"))
    .withColumn("WeekOfYear", F.weekofyear("Date"))
    .withColumn("IsWeekend", F.dayofweek("Date").isin(1, 7))
)

(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("Tables/dbo/dim_date")
)

print(f"dim_date written - {dim_date.count()} rows ({date_bounds.min_date} to {date_bounds.max_date})")


## Resource Dimension

Select the latest current inventory row for each resource and expose the stable attributes expected by the semantic model.


In [ ]:
latest_resource = Window.partitionBy("resource_id").orderBy(
    F.col("snapshot_date").desc_nulls_last(),
    F.col("effective_start_date").desc_nulls_last(),
)

dim_resource = (
    gold_resources
    .where(F.col("resource_id").isNotNull())
    .where(F.coalesce(F.col("is_current"), F.lit(True)))
    .withColumn("_row_number", F.row_number().over(latest_resource))
    .where(F.col("_row_number") == 1)
    .withColumn("is_compliant", F.col("compliance_status") == F.lit("Compliant"))
    .select(
        "resource_id",
        "resource_name",
        "resource_type",
        "resource_group",
        "region",
        "tags",
        "total_cost",
        "is_compliant",
        "effective_start_date",
        "effective_end_date",
        "is_current",
    )
)

(
    dim_resource.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("Tables/dbo/dim_resource")
)

print(f"dim_resource written - {dim_resource.count()} rows")


## Validation

Confirm that both dimensions are registered in the Lakehouse and have unique keys before creating the semantic model.


In [ ]:
for table_name, key_column in [("dim_date", "Date"), ("dim_resource", "resource_id")]:
    table = spark.read.format("delta").load(f"Tables/dbo/{table_name}")
    row_count = table.count()
    distinct_keys = table.select(key_column).distinct().count()
    if row_count == 0 or row_count != distinct_keys:
        raise ValueError(
            f"{table_name} validation failed: {row_count} rows, {distinct_keys} distinct keys"
        )
    print(f"{table_name}: {row_count} rows, unique {key_column}")

print("Semantic model dimensions are ready.")
